# Preparación y Validación del Dataset de Perfil de Cliente

Este notebook documenta el flujo reproducible de preparación de datos para la versión pública del proyecto de analítica de clientes.

## Objetivos del análisis
- Auditar la calidad del dataset original.
- Aplicar reglas explícitas de limpieza y estandarización.
- Generar un dataset analítico reutilizable en Power BI.
- Validar variables clave antes de la sincronización del dashboard.

## Preguntas de negocio que sustenta esta base analítica
1. ¿Qué segmentos educativos concentran el mayor gasto?
2. ¿Cómo influye la composición del hogar en el gasto total y en el gasto premium?
3. ¿Cómo se relaciona la antig?edad del cliente con el gasto y la actividad de compra?
4. ¿La edad modifica la preferencia entre compras web y compras en tienda?
5. ¿Qué segmentos combinan alto valor y sensibilidad a descuentos?


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f"{x:,.2f}")

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw' / 'marketing_raw.csv').exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / 'data' / 'raw' / 'marketing_raw.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'marketing_clean.csv'

assert RAW_PATH.exists(), f'Raw dataset not found at {RAW_PATH}'

print(f'Repository root: {ROOT.name}')
print(f'Raw dataset: {RAW_PATH.relative_to(ROOT)}')
print(f'Processed dataset target: {PROCESSED_PATH.relative_to(ROOT)}')


Repository root: customer-profile-analytics-powerbi
Raw dataset: data\raw\marketing_raw.csv
Processed dataset target: data\processed\marketing_clean.csv


In [2]:
df_raw = pd.read_csv(RAW_PATH)

print(f'Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
display(df_raw.head())


Raw shape: 2,205 rows x 39 columns


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Age,Customer_Days,marital_Divorced,marital_Married,marital_Single,marital_Together,marital_Widow,education_2n Cycle,education_Basic,education_Graduation,education_Master,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall
0,$58138,0,0,58,$635,$88,$546,$172,$88,$88,3,8,10,4,7,0,0,0,0,0,0,3,11,1,63,2822,0,0,1,0,0,0,0,1,0,0,$1529,1441,0
1,$46344,1,1,38,$11,$1,$6,$2,$1,$6,2,1,1,2,5,0,0,0,0,0,0,3,11,0,66,2272,0,0,1,0,0,0,0,1,0,0,$21,15,0
2,$71613,0,0,26,$426,$49,$127,$111,$21,$42,1,8,2,10,4,0,0,0,0,0,0,3,11,0,55,2471,0,0,0,1,0,0,0,1,0,0,$734,692,0
3,$26646,1,0,26,$11,$4,$20,$10,$3,$5,2,2,0,4,6,0,0,0,0,0,0,3,11,0,36,2298,0,0,0,1,0,0,0,1,0,0,$48,43,0
4,$58293,1,0,94,$173,$43,$118,$46,$27,$15,5,5,3,6,5,0,0,0,0,0,0,3,11,0,39,2320,0,1,0,0,0,0,0,0,0,1,$407,392,0


## Auditoría inicial del conjunto de datos

La revisión inicial se concentra en los puntos que más afectan la confiabilidad del análisis: duplicados, tipos de dato, valores atípicos relevantes e integridad de las variables codificadas.


In [3]:
marital_cols = [
    'marital_Single', 'marital_Married', 'marital_Together',
    'marital_Divorced', 'marital_Widow'
]

education_cols = [
    'education_2n Cycle', 'education_Basic', 'education_Graduation',
    'education_Master', 'education_PhD'
]

money_cols = [
    'Income', 'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'MntTotal'
]

duplicate_rows = int(df_raw.duplicated().sum())
age_99999_count = int((df_raw['Age'] == 99999).sum())
object_columns = df_raw.select_dtypes(include='object').columns.tolist()

raw_audit = pd.DataFrame(
    {
        'Metric': [
            'Rows', 'Columns', 'Exact duplicate rows',
            'Rows after exact deduplication', 'Null cells',
            'Age == 99999 rows', 'Object columns count'
        ],
        'Value': [
            len(df_raw),
            df_raw.shape[1],
            duplicate_rows,
            len(df_raw.drop_duplicates()),
            int(df_raw.isna().sum().sum()),
            age_99999_count,
            len(object_columns)
        ]
    }
)

display(raw_audit)
print('Object columns:', object_columns)


,Metric,Value
0,Rows,2205
1,Columns,39
2,Exact duplicate rows,184
3,Rows after exact deduplication,2021
4,Null cells,0
5,Age == 99999 rows,0
6,Object columns count,8


Object columns: ['Income', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'MntTotal']


In [4]:
marital_sum = df_raw[marital_cols].sum(axis=1)
education_sum = df_raw[education_cols].sum(axis=1)

assert marital_sum.eq(1).all(), 'Marital one-hot encoding is invalid in the raw dataset.'
assert education_sum.eq(1).all(), 'Education one-hot encoding is invalid in the raw dataset.'

print('Raw one-hot validation passed.')
print(f'Marital rows with exactly one active flag: {int(marital_sum.eq(1).sum()):,}')
print(f'Education rows with exactly one active flag: {int(education_sum.eq(1).sum()):,}')


Raw one-hot validation passed.
Marital rows with exactly one active flag: 2,205
Education rows with exactly one active flag: 2,205


## Criterios de limpieza y estandarización

Las reglas aplicadas en este flujo buscan dejar una base analítica consistente y reutilizable.

- Eliminar duplicados exactos únicamente en la versión analítica.
- Convertir a numérico las variables monetarias almacenadas como texto.
- Reconstruir variables categóricas a partir de columnas one-hot.
- Incorporar variables derivadas estables para análisis y visualización.
- Detener la exportación si falla una validación crótica.


In [5]:
def clean_currency(series: pd.Series) -> pd.Series:
    return series.replace({'[$,]': ''}, regex=True).astype('float64')

df = df_raw.drop_duplicates().reset_index(drop=True).copy()

for col in money_cols:
    df[col] = clean_currency(df[col])

marital_map = {
    'marital_Single': 'Single',
    'marital_Married': 'Married',
    'marital_Together': 'Together',
    'marital_Divorced': 'Divorced',
    'marital_Widow': 'Widow'
}

education_map = {
    'education_2n Cycle': '2n Cycle',
    'education_Basic': 'Basic',
    'education_Graduation': 'Graduation',
    'education_Master': 'Master',
    'education_PhD': 'PhD'
}

df['Marital_Status'] = df[marital_cols].idxmax(axis=1).map(marital_map)
df['Education_Level'] = df[education_cols].idxmax(axis=1).map(education_map)

assert df['Education_Level'].eq('2n Cycle').sum() == int(df['education_2n Cycle'].sum())

print(f'Rows after exact deduplication: {len(df):,}')
display(df[['Income', 'MntTotal', 'Marital_Status', 'Education_Level']].head())


Rows after exact deduplication: 2,021


,Income,MntTotal,Marital_Status,Education_Level
0,"58,138.00","1,529.00",Single,Graduation
1,"46,344.00",21.00,Single,Graduation
2,"71,613.00",734.00,Together,Graduation
3,"26,646.00",48.00,Together,Graduation
4,"58,293.00",407.00,Married,PhD


In [6]:
tenure_median_days = float(df['Customer_Days'].median())
high_value_threshold = float(df['MntTotal'].quantile(0.75))

age_bins = [0, 30, 45, 60, np.inf]
age_labels = ['Young Adults', 'Adults', 'Mid-Age', 'Seniors']

df['Has_Children'] = (df['Kidhome'] + df['Teenhome']) > 0
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=False).astype('string')
df['Customer_Tenure_Years'] = (df['Customer_Days'] / 365.25).round(2)
df['Customer_Tenure_Group'] = np.where(
    df['Customer_Days'] < tenure_median_days,
    'Newer',
    'Established'
)
df['Is_High_Value_Customer'] = df['MntTotal'] >= high_value_threshold
df['Premium_Spend_Share'] = np.where(
    df['MntTotal'] > 0,
    df['MntGoldProds'] / df['MntTotal'],
    0.0
)
df['Premium_Spend_Share'] = df['Premium_Spend_Share'].round(4)

feature_summary = pd.DataFrame(
    {
        'Feature': [
            'Age_Group bins',
            'Customer_Tenure_Group threshold (days)',
            'High-value threshold (75th percentile MntTotal)'
        ],
        'Definition': [
            '[0, 30), [30, 45), [45, 60), [60, +inf)',
            tenure_median_days,
            round(high_value_threshold, 2)
        ]
    }
)

display(feature_summary)


,Feature,Definition
0,Age_Group bins,"[0, 30), [30, 45), [45, 60), [60, +inf)"
1,Customer_Tenure_Group threshold (days),"2,511.00"
2,High-value threshold (75th percentile MntTotal),964.00


In [7]:
df_clean = df.drop(columns=marital_cols + education_cols).copy()

preferred_order = [
    'Age', 'Age_Group', 'Income', 'Marital_Status', 'Education_Level',
    'Kidhome', 'Teenhome', 'Has_Children', 'Customer_Days',
    'Customer_Tenure_Years', 'Customer_Tenure_Group', 'Recency',
    'NumWebPurchases', 'NumStorePurchases', 'NumCatalogPurchases',
    'NumDealsPurchases', 'NumWebVisitsMonth', 'MntTotal', 'MntGoldProds',
    'Premium_Spend_Share', 'Is_High_Value_Customer'
]

remaining_cols = [col for col in df_clean.columns if col not in preferred_order]
df_clean = df_clean[preferred_order + remaining_cols]

display(df_clean.head())
print(f'Clean dataset columns: {len(df_clean.columns)}')


,Age,Age_Group,Income,Marital_Status,Education_Level,Kidhome,Teenhome,Has_Children,Customer_Days,Customer_Tenure_Years,Customer_Tenure_Group,Recency,NumWebPurchases,NumStorePurchases,NumCatalogPurchases,NumDealsPurchases,NumWebVisitsMonth,MntTotal,MntGoldProds,Premium_Spend_Share,Is_High_Value_Customer,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,MntRegularProds,AcceptedCmpOverall
0,63,Seniors,"58,138.00",Single,Graduation,0,0,False,2822,7.73,Established,58,8,4,10,3,7,"1,529.00",88.00,0.06,True,635.00,88.00,546.00,172.00,88.00,0,0,0,0,0,0,3,11,1,1441,0
1,66,Seniors,"46,344.00",Single,Graduation,1,1,True,2272,6.22,Newer,38,1,2,1,2,5,21.00,6.00,0.29,False,11.00,1.00,6.00,2.00,1.00,0,0,0,0,0,0,3,11,0,15,0
2,55,Mid-Age,"71,613.00",Together,Graduation,0,0,False,2471,6.77,Newer,26,8,10,2,1,4,734.00,42.00,0.06,False,426.00,49.00,127.00,111.00,21.00,0,0,0,0,0,0,3,11,0,692,0
3,36,Adults,"26,646.00",Together,Graduation,1,0,True,2298,6.29,Newer,26,2,4,0,2,6,48.00,5.00,0.10,False,11.00,4.00,20.00,10.00,3.00,0,0,0,0,0,0,3,11,0,43,0
4,39,Adults,"58,293.00",Married,PhD,1,0,True,2320,6.35,Newer,94,5,6,3,5,5,407.00,15.00,0.04,False,173.00,43.00,118.00,46.00,27.00,0,0,0,0,0,0,3,11,0,392,0


Clean dataset columns: 37


## Validación y exportación del dataset analítico

Antes de exportar, se verifican integridad, consistencia tipológica y ausencia de ambiguedades en las variables derivadas. El objetivo es que el dataset resultante pueda reutilizarse en Power BI sin ajustes manuales adicionales.


In [8]:
expected_numeric_cols = money_cols + [
    'Customer_Days', 'Customer_Tenure_Years', 'Recency',
    'NumWebPurchases', 'NumStorePurchases', 'NumCatalogPurchases',
    'NumDealsPurchases', 'NumWebVisitsMonth', 'Premium_Spend_Share'
]

assert len(df_clean) == 2021, f'Expected 2021 analytical rows, found {len(df_clean)}.'
assert not df_clean.duplicated().any(), 'Exact duplicates remain in the clean dataset.'
assert all(pd.api.types.is_numeric_dtype(df_clean[col]) for col in expected_numeric_cols), 'Numeric columns are not fully numeric.'
assert df_clean['Education_Level'].nunique() == 5, 'Education reconstruction is incomplete.'
assert df_clean['Marital_Status'].nunique() == 5, 'Marital reconstruction is incomplete.'
assert (df_clean['Age'] != 99999).all(), 'Invalid age value found in clean dataset.'
assert df_clean[['Marital_Status', 'Education_Level', 'Age_Group', 'Customer_Tenure_Group']].isna().sum().sum() == 0, 'Derived categorical fields contain nulls.'
assert int(df_clean.isna().sum().sum()) == 0, 'The clean dataset contains null values.'

validation_summary = pd.DataFrame(
    {
        'Check': [
            'Analytical rows', 'Exact duplicates removed', 'Money columns numeric',
            'Education includes 2n Cycle', 'Marital categories reconstructed',
            'No Age == 99999', 'No nulls introduced'
        ],
        'Result': [
            len(df_clean),
            True,
            True,
            '2n Cycle' in df_clean['Education_Level'].unique(),
            df_clean['Marital_Status'].nunique() == 5,
            True,
            True
        ]
    }
)

display(validation_summary)

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_PATH, index=False)

reloaded = pd.read_csv(PROCESSED_PATH)
assert reloaded.shape == df_clean.shape, 'Reloaded dataset shape does not match exported dataset.'
assert list(reloaded.columns) == list(df_clean.columns), 'Reloaded dataset schema does not match exported dataset.'

print(f'Processed dataset exported to: {PROCESSED_PATH.relative_to(ROOT)}')
print(f'Exported shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')


,Check,Result
0,Analytical rows,2021
1,Exact duplicates removed,True
2,Money columns numeric,True
3,Education includes 2n Cycle,True
4,Marital categories reconstructed,True
5,No Age == 99999,True
6,No nulls introduced,True


Processed dataset exported to: data\processed\marketing_clean.csv
Exported shape: 2,021 rows x 37 columns


## Indicadores de control analítico

Estas métricas de control permiten comprobar que el dataset limpio ya soporta las principales preguntas de negocio y que la base queda lista para la siguiente etapa de visualización y storytelling.


In [9]:
kpi_summary = pd.DataFrame(
    {
        'Metric': [
            'Average Income',
            'Average Total Spend',
            'Customers With Children (%)',
            'High-Value Customer Rate (%)',
            'Average Premium Spend Share'
        ],
        'Value': [
            round(df_clean['Income'].mean(), 2),
            round(df_clean['MntTotal'].mean(), 2),
            round(df_clean['Has_Children'].mean() * 100, 2),
            round(df_clean['Is_High_Value_Customer'].mean() * 100, 2),
            round(df_clean['Premium_Spend_Share'].mean(), 4)
        ]
    }
)

spend_by_education = (
    df_clean.groupby('Education_Level', observed=False)['MntTotal']
    .mean()
    .round(2)
    .sort_values(ascending=False)
    .rename('Average Total Spend')
    .to_frame()
)

premium_by_household = (
    df_clean.groupby('Has_Children')['Premium_Spend_Share']
    .mean()
    .round(4)
    .rename('Average Premium Spend Share')
    .to_frame()
)

tenure_summary = (
    df_clean.groupby('Customer_Tenure_Group')['MntTotal']
    .agg(['mean', 'median'])
    .round(2)
    .rename(columns={'mean': 'Average Spend', 'median': 'Median Spend'})
)

channel_mix_by_age = (
    df_clean.groupby('Age_Group', observed=False)[['NumWebPurchases', 'NumStorePurchases']]
    .sum()
)
channel_mix_by_age = (
    channel_mix_by_age.div(channel_mix_by_age.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

discount_sensitivity = (
    df_clean.groupby('Is_High_Value_Customer')['NumDealsPurchases']
    .mean()
    .round(2)
    .rename('Average Discount Purchases')
    .to_frame()
)

display(kpi_summary)
display(spend_by_education)
display(premium_by_household)
display(tenure_summary)
display(channel_mix_by_age)
display(discount_sensitivity)


,Metric,Value
0,Average Income,"51,687.26"
1,Average Total Spend,563.79
2,Customers With Children (%),71.94
3,High-Value Customer Rate (%),25.04
4,Average Premium Spend Share,0.16


,Average Total Spend
Education_Level,
PhD,631.53
Master,579.23
Graduation,573.53
2n Cycle,453.57
Basic,61.16


,Average Premium Spend Share
Has_Children,
False,0.12
True,0.18


,Average Spend,Median Spend
Customer_Tenure_Group,,
Established,643.24,483.00
Newer,484.26,224.00


,NumWebPurchases,NumStorePurchases
Age_Group,,
Adults,40.39,59.61
Mid-Age,42.28,57.72
Seniors,41.58,58.42
Young Adults,37.18,62.82


,Average Discount Purchases
Is_High_Value_Customer,
False,2.50
True,1.81


## Resultado del procesamiento

El proyecto queda con una única base analítica documentada y reproducible:
- Entrada original en `data/raw/marketing_raw.csv`.
- Salida analítica en `data/processed/marketing_clean.csv`.
- Lógica de negocio consistente para su uso posterior en Power BI.

Con esta base, el siguiente paso es alinear el dashboard y la narrativa del portafolio sobre definiciones ya validadas.
